# RAG-SQL-Layer demo

Ask a question in plain language. Each run shows the model's **thinking**, the **SQL** it wrote, the **SQL output**, and the **answer**.

Before running: `docker compose up -d`, Ollama reachable over Tailscale, and `uv run rag-sql-index` done once.

In [1]:
from rag_sql.display import run_and_display

In [2]:
run_and_display("Which department has the highest average salary?")

**Question:** Which department has the highest average salary?

### SQL
```sql
SELECT department,
       round(avg(salary), 2) AS highest_avg_salary
FROM employees
WHERE department IS NOT NULL AND salary IS NOT NULL
GROUP BY department
ORDER BY highest_avg_salary DESC
LIMIT 1;
```

### SQL output
1 row(s)

,department,highest_avg_salary
0,HR,61699.09


### Answer
The HR department has the highest average salary at 61699.09.

In [3]:
run_and_display("How many remote employees are in each city?")

**Question:** How many remote employees are in each city?

### SQL
```sql
SELECT
    city,
    COUNT(*) FILTER (WHERE remote_work) AS remote_employee_count
FROM
    employees
GROUP BY
    city
ORDER BY
    remote_employee_count DESC;
```

### SQL output
5 row(s)

,city,remote_employee_count
0,Pune,36
1,Delhi,28
2,Bangalore,27
3,Mumbai,17
4,Chennai,15


### Answer
Pune: 36, Delhi: 28, Bangalore: 27, Mumbai: 17, Chennai: 15.

In [4]:
run_and_display("Who are the top 3 performers in Sales by salary?")

**Question:** Who are the top 3 performers in Sales by salary?

### SQL
```sql
SELECT emp_name, department, salary, performance_rating
FROM employees
WHERE department = 'Sales' AND salary IS NOT NULL
ORDER BY salary DESC
LIMIT 3;
```

### SQL output
3 row(s)

,emp_name,department,salary,performance_rating
0,Employee_156,Sales,93771.08,NaN
1,Employee_263,Sales,87416.23,3.0
2,Employee_260,Sales,86638.99,5.0


### Answer
The top 3 performers in Sales by salary are:
1. Employee_156: 93771.08
2. Employee_263: 87416.23
3. Employee_260: 86638.99

## Follow-up questions

A `Chat` keeps one conversation. Follow-ups are rewritten into standalone questions using the earlier turns (shown as *Interpreted as*), and every turn is saved in Postgres. To continue a conversation later, pass its thread id: `Chat("<thread id>")`.

In [1]:
from rag_sql.display import Chat, show_threads

chat = Chat()
chat

Chat(thread_id='b5d8c4e6e17c4bc9bb71149f13978297')

In [2]:
chat.ask("Which department has the highest average salary?")

**Question:** Which department has the highest average salary?

### SQL
```sql
SELECT department,
       round(avg(salary), 2) AS avg_salary
FROM employees
WHERE department IS NOT NULL AND salary IS NOT NULL
GROUP BY department
ORDER BY avg_salary DESC
LIMIT 1;
```

### SQL output
1 row(s)

,department,avg_salary
0,HR,61699.09


### Answer
The HR department has the highest average salary at 61699.09.

In [3]:
chat.ask("And which one has the lowest?")

**Question:** And which one has the lowest?

### SQL
```sql
SELECT
  department,
  ROUND(CAST(AVG(salary) AS DECIMAL), 2) AS avg_salary
FROM employees
WHERE
  department IS NOT NULL AND salary IS NOT NULL
GROUP BY
  department
ORDER BY
  avg_salary ASC
LIMIT 1
```

### SQL output
1 row(s)

,department,avg_salary
0,Sales,56721.77


### Answer
The Sales department has the lowest average salary at 56721.77.

In [4]:
chat.ask("List the 5 highest paid employees in that department")

**Question:** List the 5 highest paid employees in that department

### SQL
```sql
SELECT
  emp_name,
  department,
  salary
FROM employees
WHERE
  department = 'Sales' AND salary IS NOT NULL
ORDER BY
  salary DESC
LIMIT 5;
```

### SQL output
5 row(s)

,emp_name,department,salary
0,Employee_156,Sales,93771.08
1,Employee_263,Sales,87416.23
2,Employee_260,Sales,86638.99
3,Employee_182,Sales,82998.18
4,Employee_168,Sales,82280.69


### Answer
The 5 highest paid employees in the Sales department are:
1. Employee_156: 93771.08
2. Employee_263: 87416.23
3. Employee_260: 86638.99
4. Employee_182: 82998.18
5. Employee_168: 82280.69

In [5]:
chat.show_history()

,question,standalone,sql,row_count,answer,error
0,Which department has the highest average salary?,Which department has the highest average salary?,"SELECT\n department,\n ROUND(CAST(AVG(salary...",1,The HR department has the highest average sala...,None
1,And which one has the lowest?,Which department has the lowest average salary?,"SELECT\n department,\n ROUND(CAST(AVG(salary...",1,The Sales department has the lowest average sa...,None
2,List the 5 highest paid employees in that depa...,List the 5 highest paid employees in the Sales...,"SELECT\n emp_name,\n department,\n salary\n...",5,The 5 highest paid employees in the Sales depa...,None


In [6]:
show_threads()

,thread_id,turns,first_question,last_at
0,b5d8c4e6e17c4bc9bb71149f13978297,3,Which department has the highest average salary?,2026-09-24 03:09:49.044174+00:00
